# 04 — Sales Forecast Analysis
**Prophet + Monthly Seasonality (v2)**

Fixed: disabled yearly seasonality (only 24 months), added manual monthly seasonality.

## Setup

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from config import PG_URL
engine = create_engine(PG_URL)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('Blues_d')
print("Setup complete")

from prophet.serialize import model_from_json
from config import MODELS_DIR


## 1. Load model and data

In [ ]:

sql = '''SELECT DATE_TRUNC('month',order_date::date) AS ds, ROUND(SUM(revenue)::numeric,2) AS y
FROM olist.fact_orders GROUP BY DATE_TRUNC('month',order_date::date) ORDER BY ds'''
with engine.connect() as conn:
    df = pd.read_sql(text(sql),conn)
df['ds'] = pd.to_datetime(df['ds']).dt.tz_localize(None)
df['y'] = df['y'].astype(float)
with open(os.path.join(MODELS_DIR,'forecast_model.json'),'r') as f:
    model = model_from_json(f.read())
future = model.make_future_dataframe(periods=6,freq='MS')
forecast = model.predict(future)
print(f"Historical: {len(df)} months")
print(f"Date range: {df['ds'].min().date()} to {df['ds'].max().date()}")
print(f"Avg monthly revenue: ${df['y'].mean():,.2f}")


## 2. Forecast visualization with confidence intervals

In [ ]:

fig,ax = plt.subplots(figsize=(13,5))
ax.fill_between(forecast['ds'],forecast['yhat_lower'],forecast['yhat_upper'],alpha=0.2,color='steelblue',label='95% CI')
ax.plot(forecast['ds'],forecast['yhat'],color='steelblue',lw=2,label='Forecast')
ax.scatter(df['ds'],df['y'],color='black',s=20,zorder=5,label='Actual')
cutoff = df['ds'].max()
ax.axvline(cutoff,color='red',linestyle='--',lw=1,label='Forecast start')
ax.set_title('Revenue Forecast — Prophet v2 (monthly seasonality)')
ax.set_ylabel('Revenue ($)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x/1e6:.1f}M'))
ax.legend(); plt.tight_layout(); plt.show()
future_only = forecast[forecast['ds']>cutoff][['ds','yhat','yhat_lower','yhat_upper']]
print("6-Month Forecast:")
for _,r in future_only.iterrows():
    print(f"  {r['ds'].strftime('%Y-%m')}  ->  ${r['yhat']:>12,.2f}  [${r['yhat_lower']:,.2f} – ${r['yhat_upper']:,.2f}]")


## 3. Actual vs predicted

In [ ]:

hist = forecast[forecast['ds'].isin(df['ds'])].merge(df,on='ds')
hist['error_pct'] = abs(hist['yhat']-hist['y'])/hist['y']*100
fig,axes = plt.subplots(1,2,figsize=(14,5))
axes[0].plot(hist['ds'],hist['y']/1e6,label='Actual',color='black',lw=2)
axes[0].plot(hist['ds'],hist['yhat']/1e6,label='Predicted',color='steelblue',lw=2,linestyle='--')
axes[0].set_title('Actual vs Predicted Revenue'); axes[0].set_ylabel('Revenue ($M)'); axes[0].legend()
axes[1].bar(hist['ds'],hist['error_pct'],color='steelblue',alpha=0.7)
axes[1].set_title('Prediction Error % by Month'); axes[1].set_ylabel('Error %')
plt.tight_layout(); plt.show()
print(f"Note: High MAPE due to only 24 months of training data.")


## 4. Seasonality components

In [ ]:

fig = model.plot_components(forecast)
fig.suptitle('Prophet Model Components',y=1.02)
plt.tight_layout(); plt.show()
